# 1. Analysis environment

This notebook explains how to build the Python environment in which every notebook of this repository was run, and then checks it.
Run the commands of section 1.2 in a terminal **before** opening the notebooks, then open this notebook with the new kernel and run it.

## 1.1 What the analysis was run on

| Item | Version or size |
|---|---|
| Operating system | Windows 11 (64-bit). Linux works with the same commands (see 1.2). |
| GPU | NVIDIA GeForce RTX 5090, 32 GB; CUDA 12.8 build of PyTorch. The analysis used at most 9 GB of GPU memory (`GPU_MEMORY_FRACTION` in notebook 00). |
| Python | 3.12.16 (managed by [uv](https://docs.astral.sh/uv/)) |
| Main packages | PyTorch 2.11.0, NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, scikit-learn 1.9.1, transformers 4.57.6 (exact pins in `requirements.txt`) |
| Memory | 64 GB RAM recommended (32 GB is enough if `CPU_JOBS` in notebook 00 is lowered to 4) |
| Disk | raw data: EchoNext 24 GB; MIMIC-IV-ECG 90 GB if downloaded in full, or about 27 GB for only the records used; the other MIMIC-IV files 0.3 GB. Derived: the MIMIC-IV waveform array 26 GB, embeddings 7 GB, model weights 2 GB |

Running time on this computer: notebook 02 under a minute; 03 about 17 minutes (reading the MIMIC-IV waveforms takes 14); 04 about 1.5 hours (the embeddings, estimated from a timed sample: about 25 minutes for EchoNext and under an hour for MIMIC-IV); 05 about 40 minutes; 06 about 10 minutes; 07 and 08 about 15 minutes each; 09, 10 and 11 one or two minutes each.

## 1.2 Create the environment (terminal)

Install `uv` first ([instructions](https://docs.astral.sh/uv/getting-started/installation/)). Then, in the folder of this repository:

**Windows (PowerShell)**
```powershell
uv venv --python 3.12 .venv
.venv\Scripts\Activate.ps1
uv pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
uv pip install -r requirements.txt
python -m ipykernel install --user --name ecgfm-race --display-name "ECG-FM race encoding (Python 3.12)"
jupyter lab
```

**Linux or macOS (bash)**
```bash
uv venv --python 3.12 .venv
source .venv/bin/activate
uv pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128   # CPU only: --index-url https://download.pytorch.org/whl/cpu
uv pip install -r requirements.txt
python -m ipykernel install --user --name ecgfm-race --display-name "ECG-FM race encoding (Python 3.12)"
jupyter lab
```

PyTorch is installed in its own step because its CUDA builds come from the PyTorch package index. In Jupyter, choose the kernel **ECG-FM race encoding (Python 3.12)** for every notebook (Kernel → Change Kernel).

To run a notebook without opening it (useful for the long ones), use for example:
```bash
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output executed/05_race_probes.ipynb 05_race_probes.ipynb
```

## 1.3 Order of the notebooks

| Notebook | What it does | Paper items it feeds |
|---|---|---|
| 00_settings | global settings and shared functions (run by every notebook) | |
| 01_environment_setup | this notebook | |
| 02_data_echonext | EchoNext: download, cohort, labels, five-fold design | Table 1, Supplementary Figs. 1 and 2 |
| 03_data_mimic | MIMIC-IV: download, cohort, labels, waveforms, five-fold design, conversion to the EchoNext format | Table 1, Supplementary Fig. 1, Supplementary Table 2 |
| 04_foundation_model_embeddings | download the three models; extract one frozen embedding per ECG | Supplementary Table 1 |
| 05_race_probes | question 1: is race decodable from the embeddings? | Fig. 1b, Supplementary Table 3, Table 2 |
| 06_cardiac_structure | question 2: is the race signal explained by cardiac structure? | Fig. 1c, Supplementary Table 4, Table 2 |
| 07_disease_detectors | question 3: detectors for every diagnosis and their errors by group | Fig. 2, Supplementary Tables 5 and 6, Table 2 |
| 08_race_erasure | question 4: erasing race information (LEACE) | Figs. 3 and 4, Supplementary Tables 3 and 7, Table 2 |
| 09_prevalence_equalised_refit | refitting composite SHD detectors after equalising prevalence | Supplementary Fig. 3, Table 2 |
| 10_main_tables_and_figures | Tables 1 and 2, Figs. 1–4 | |
| 11_supplementary_tables_and_figures | Supplementary Figs. 1–3, Supplementary Tables 1–7 | |

Each notebook can be rerun on its own once the notebooks before it have been run: everything it needs is read from `WORK_DIR` and `RESULTS_DIR`.

## 1.4 Check the installed packages

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python", sys.version.split()[0], "(analysis: 3.12.16)")
expected = {"torch": "2.11.0", "numpy": "2.5.4", "pandas": "3.1.1", "pyarrow": "26.0.1", "scipy": "1.18.1", "scikit-learn": "1.9.1",
            "threadpoolctl": "3.8.0", "joblib": "1.6.0", "matplotlib": "3.11.2", "transformers": "4.57.6", "safetensors": "0.8.0",
            "huggingface-hub": "0.36.2", "requests": "2.34.2", "ipywidgets": "8.1.9"}
rows = []
for pkg, want in expected.items():
    try:
        have = version(pkg)
    except PackageNotFoundError:
        have = "not installed"
    rows.append((pkg, want, have, "ok" if have.split("+")[0] == want else "DIFFERENT"))
w = max(len(p) for p in expected)
for pkg, want, have, ok in rows:
    print(f"{pkg:<{w}}  expected {want:<8}  installed {have:<16} {ok}")
if any(r[3] != "ok" for r in rows):
    print("\nSome versions differ. The notebooks may still run, but numbers can differ in the last digits; install requirements.txt to reproduce exactly.")

## 1.5 Check the GPU

The embeddings (notebook 04) need a CUDA GPU in practice; the probes run on the GPU too, and fall back to the CPU (slowly) if there is none.

In [ ]:
import torch

print("PyTorch", torch.__version__, "| CUDA build", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU 0: {p.name}, {p.total_memory / 2**30:.1f} GB")
    x = torch.randn(1000, 1000, device="cuda", dtype=torch.float64)
    print("float64 matrix product on the GPU:", bool(torch.isfinite(x @ x).all()))

## 1.6 Load the global settings and create the folders

This runs notebook 00 (as every other notebook does) and creates the folder tree under `MASTER_DIR`. Edit the first code cell of `00_settings.ipynb` if the paths are not where you want them.

In [ ]:
%run ./00_settings.ipynb

for name, p in [("MASTER_DIR", MASTER_DIR), ("DATA_DIR", DATA_DIR), ("MODEL_DIR", MODEL_DIR), ("WORK_DIR", WORK_DIR), ("RESULTS_DIR", RESULTS_DIR),
                ("OUTPUT_DIR", OUTPUT_DIR)]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"{name:<12} {p}")
print()
for name, p in [("ECHONEXT_DIR", ECHONEXT_DIR), ("MIMIC_IV_DIR", MIMIC_IV_DIR), ("MIMIC_ED_DIR", MIMIC_ED_DIR), ("MIMIC_ECG_DIR", MIMIC_ECG_DIR),
                ("MIMIC_ECHO_DIR", MIMIC_ECHO_DIR)]:
    print(f"{name:<15} {'found    ' if p.exists() else 'not found'} {p}")
print("\nRaw data that are not found yet are downloaded in notebooks 02 and 03.")